## 0. GPU Verification

It is mandatory to verify that a GPU is available, as Foundation Models (Chronos, TimesFM) require it for efficient execution. If no GPU is detected, these models will be excluded from the benchmark.

In [1]:
import torch
if torch.cuda.is_available():
    print(f"✅ GPU Detected: {torch.cuda.get_device_name(0)}")
else:
    print("❌ NO GPU DETECTED. Foundation models will run on CPU (very slow) or be skipped.")

✅ GPU Detected: Tesla T4


# Evaluating Zero-Shot Foundation Models for Epidemiological Forecasting of Influenza-Like Illness in Italy

This notebook is designed to be a fully automated, "plug-and-play" environment to reproduce the benchmarking results of my thesis. It handles the environment setup, data ingestion from the official Influcast repository, and the execution of the zero-shot foundation models.

### Recommendation for Google Colab

Before starting, please ensure you are using a GPU Runtime:

1. Go to Runtime -> Change runtime type.
2. Select T4 GPU (or better).

## 1. Environment setup

As a first step we clone the github repo containg our codebase, and install the dependencies

In [2]:
%cd /content

# 1. clone the repo
REPO_URL = f"https://github.com/biagio-incardona/Biagio-Incardona-Master-thesis.git"
!git clone -b TimeGPT {REPO_URL}

/content
Cloning into 'Biagio-Incardona-Master-thesis'...
remote: Enumerating objects: 284, done.
remote: Counting objects: 100% (284/284), done.
remote: Compressing objects: 100% (237/237), done.
remote: Total 284 (delta 72), reused 251 (delta 41), pack-reused 0 (from 0)
Receiving objects: 100% (284/284), 1.19 MiB | 18.23 MiB/s, done.
Resolving deltas: 100% (72/72), done.


In [3]:

# 2. install dependencies
%cd Biagio-Incardona-Master-thesis
!grep -vE "timesfm|tirex" requirements.txt > requirements_core.txt
!pip install -r requirements_core.txt

# 3. Install the foundation models directly from GitHub (force upgrade)
!pip install -U "timesfm[torch] @ git+https://github.com/google-research/timesfm.git"
!pip install git+https://github.com/NX-AI/tirex.git


# 3. Setup Python Path to recognize our 'src' directory

import sys
import os



sys.path.append(os.getcwd())
try:
    from google.colab import userdata
    os.environ["TIMEGPT_TOKEN"] = userdata.get('TimeGPT')
except:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    os.environ["TIMEGPT_TOKEN"] = user_secrets.get_secret("TimeGPT")

/content/Biagio-Incardona-Master-thesis
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 689.1/689.1 kB 15.7 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 354.6/354.6 kB 20.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.3/37.3 MB 50.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 128.1/128.1 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.6/46.6 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.1/80.1 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 101.8 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.9/51.9 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 348.2/348.2 kB 24.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 281.0/281.0 kB 22.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.9/59.9 kB 

## 2. Data Ingestion & Preprocessing

We ingest raw **ILI** (Influenza-Like Illness) data from the `Influcast` repository and historical records. It is important to distinguish this from **ARI** (Acute Respiratory Infections), which has been introduced in more recent surveillance protocols (2024/25+) but is not the primary focus of this historical evaluation.

The preprocessing pipeline ensures:
   * **Epidemic Indexing**: To handle the seasonal nature of surveillance, we use an *epidemic time index* that concatenates observed weeks. This avoids artificial zero-filling during off-season periods.
   * **Metadata Preservation**: The actual calendar dates are preserved in a `calendar_ds` column to ensure proper mapping during visualization.
   * **Standardization**: All records are mapped to ISO weeks and Sunday-based dates.

  Columns:
   * **ds**: Synthetic date (weekly frequency) used for modeling.
   * **calendar_ds**: Real calendar Sunday of the observation.
   * **region**: Lowercase location name.
   * **y**: ILI incidence per 1,000 inhabitants.

In [4]:
import pandas as pd
from IPython.display import display

print("Ingesting raw ILI data from Influcast...")
!python3 src/data/ingestion.py

print("\nPreprocessing data into standardized format (Epidemic Indexing)...")
!python3 src/data/preprocessing.py --time-index epidemic

# --- PROFESSOR GUIDELINE: SOURCE TRACKING ---
if os.path.exists('data/processed/source_files_index.csv'):
    print("\n--- SOURCE AUDIT MANIFEST (First 10 entries) ---")
    manifest = pd.read_csv('data/processed/source_files_index.csv')
    display(manifest.head(10))

# --- PROFESSOR GUIDELINE: FULL REGIONAL INGESTION ---
if os.path.exists('data/processed/ili_gold.csv'):
    df = pd.read_csv('data/processed/ili_gold.csv')
    regions = df['region'].unique().tolist()
    print(f"\n✅ DATA READINESS VERIFIED")
    print(f"Total Regions Discovered: {len(regions)}")
    print(f"Total Observation Weeks: {len(df)}")
    print(f"Last date in series: {df['ds'].max()}")
    if len(regions) >= 22:
        print("⭐ Full panel of 22 geographic entities successfully ingested.")
else:
    print("\n❌ ERROR: Data preparation failed. Please check the ingestion logs.")

Ingesting raw ILI data from Influcast...
2026-06-19 21:06:03,174 - INFO - Cloning repository https://github.com/Predizioni-Epidemiologiche-Italia/Influcast.git into data/influcast_repo...
Cloning into 'data/influcast_repo'...
remote: Enumerating objects: 8938, done.
remote: Counting objects: 100% (8938/8938), done.
remote: Compressing objects: 100% (7937/7937), done.
remote: Total 8938 (delta 1019), reused 8837 (delta 982), pack-reused 0 (from 0)
Receiving objects: 100% (8938/8938), 10.88 MiB | 22.29 MiB/s, done.
Resolving deltas: 100% (1019/1019), done.
2026-06-19 21:06:05,737 - INFO - Season 2003-2004 | Region italia | Fallback to italia-2004_17-ILI.csv
2026-06-19 21:06:05,738 - INFO - Season 2004-2005 | Region italia | Fallback to italia-2005_16-ILI.csv
2026-06-19 21:06:05,738 - INFO - Season 2005-2006 | Region italia | Fallback to italia-2006_17-ILI.csv
2026-06-19 21:06:05,739 - INFO - Season 2006-2007 | Region italia | Fallback to italia-2007_17-ILI.csv
2026-06-19 21:06:05,739 - I

,season,region,file_used,source_type
0,2003-2004,italia,italia-2004_17-ILI.csv,weekly_fallback
1,2004-2005,italia,italia-2005_16-ILI.csv,weekly_fallback
2,2005-2006,italia,italia-2006_17-ILI.csv,weekly_fallback
3,2006-2007,italia,italia-2007_17-ILI.csv,weekly_fallback
4,2007-2008,italia,italia-2008_17-ILI.csv,weekly_fallback
5,2008-2009,italia,italia-2009_17-ILI.csv,weekly_fallback
6,2009-2010,italia,italia-2010_15-ILI.csv,weekly_fallback
7,2010-2011,italia,italia-2011_17-ILI.csv,weekly_fallback
8,2011-2012,emilia_romagna,emilia_romagna-2012_17-ILI.csv,weekly_fallback
9,2011-2012,sicilia,sicilia-2012_17-ILI.csv,weekly_fallback



✅ DATA READINESS VERIFIED
Total Regions Discovered: 22
Total Observation Weeks: 7787
Last date in series: 2018-08-19
⭐ Full panel of 22 geographic entities successfully ingested.


## Step 3: Running the Full Benchmarking Suite

Now we run the full models suite. To make this a fair academic comparison, we evaluate both **Classical Baselines**, **Advanced ML**, and the latest **Foundation Models**.

We are testing for the following models:

* **Simple Baselines**: Naive, SeasonalNaive, Drift, MovingAverage
* **Statistical**: ETS, ARIMA, SARIMA, Prophet
* **Machine Learning**: LightGBM, XGBoost, CatBoost, Ridge
* **Foundation Models**: Chronos, TimesFM, TiRex, TimeGPT

To avoid memory issues in Colab, the benchmarking is split into stages. We use dynamic origin generation (rolling window backtest) with a standard step size.

Let's start with Statistical Baselines


In [5]:
print(">>> 1. Running Classical Baselines (SARIMA, ARIMA, Prophet, etc.)...")
!python3 benchmark_ili_national.py --model Naive
!python3 benchmark_ili_national.py --model SeasonalNaive --append
!python3 benchmark_ili_national.py --model Drift --append
!python3 benchmark_ili_national.py --model MovingAverage --append
!python3 benchmark_ili_national.py --model ETS --append
!python3 benchmark_ili_national.py --model ARIMA --append
!python3 benchmark_ili_national.py --model SARIMA --append
!python3 benchmark_ili_national.py --model Prophet --append

>>> 1. Running Classical Baselines (SARIMA, ARIMA, Prophet, etc.)...
Loading data...
Loaded 615 weeks of national data.
Total origins to evaluate: 113
Horizons: [1, 2, 4, 8]

RUNNING BACKTEST FOR: Naive
DEBUG: kwargs={}
Parallelism: n_jobs=-1
Saved Naive forecasts to results/national/naive_forecasts.csv

Consolidating all results...
Calculating metrics...
Calculating metrics by origin...
Calculating seasonal peak metrics...
Run report saved to results/national/RUN_REPORT.md
Generating visualizations...
Visualizations saved to results/national/plots
Peak Metrics: results/national/all_models_peak_metrics.csv

########################################
NATIONAL ILI BENCHMARKING COMPLETE
########################################
Forecasts: results/national/all_models_forecasts.csv
Metrics:   results/national/all_models_metrics.csv

Overall Summary (MAE/WIS averaged):
            MAE       WIS
model                    
Naive  2.106809  1.488661
Loading data...
Loaded 615 weeks of national data

now let's move to ML Baselines. This set of models has been optimized using the Optuna framework for Hyperparameters Tuning.


### What is Optuna?

Optuna is an automated hyperparameter optimization framework that uses a Bayesian approach to find the most effective model configurations. Unlike traditional Grid Search, which exhaustively checks every
  combination, Optuna employs the Tree-structured Parzen Estimator (TPE) algorithm to learn from the results of previous trials. It strategically samples the search space to focus on areas that are most
  likely to improve the model's performance. In this project, it is used to fine-tune the LightGBM and XGBoost models, allowing for an efficient search of parameters like learning rates and tree depth while
  minimizing computational cost through its built-in early-stopping (pruning) mechanisms.


### Training & Benchmarking

In [ ]:
print(">>> 2. Running ML Baselines (LightGBM, XGBoost, CatBoost, Ridge)...")
!python3 benchmark_ili_national.py --model Ridge --append
!python3 benchmark_ili_national.py --model LightGBM --tune --append
!python3 benchmark_ili_national.py --model XGBoost --tune --append
!python3 benchmark_ili_national.py --model CatBoost --tune --append

>>> 2. Running ML Baselines (LightGBM, XGBoost, CatBoost, Ridge)...
Loading data...
Loaded 615 weeks of national data.
Total origins to evaluate: 113
Horizons: [1, 2, 4, 8]
Loading existing forecasts from results/national/all_models_forecasts.csv for appending...
Loaded 73224 existing forecast rows.

RUNNING BACKTEST FOR: Ridge
DEBUG: kwargs={}
Parallelism: n_jobs=-1
Backtesting: 100%|████████████████████████████| 113/113 [00:28<00:00,  4.01it/s]
Saved Ridge forecasts to results/national/ridge_forecasts.csv

Consolidating all results...
Merging with existing results. Overwriting models: ['Ridge']
Calculating metrics...
Calculating metrics by origin...
Calculating seasonal peak metrics...
Run report saved to results/national/RUN_REPORT.md
Generating visualizations...
Visualizations saved to results/national/plots
Peak Metrics: results/national/all_models_peak_metrics.csv

########################################
NATIONAL ILI BENCHMARKING COMPLETE
########################################

And now let's move to the Fundational models

In [ ]:
import torch
if not torch.cuda.is_available():
    print("WARNING: GPU not detected. Foundation models will be extremely slow on CPU.")

print(">>> 3. Running Foundation Models (Chronos, TimesFM, TiRex, TimeGPT)...")
!python3 benchmark_ili_national.py --model Chronos --model-size small --append
!python3 benchmark_ili_national.py --model Chronos --model-size v2 --append
!python3 benchmark_ili_national.py --model Chronos --model-size bolt-small --append
!python3 benchmark_ili_national.py --model TimesFM --append
!python3 benchmark_ili_national.py --model TiRex --append
!python3 benchmark_ili_national.py --model TimeGPT --append

# 4. Results Visualization

Having ran the benchmarks, we can visualize and analyse the results.

This will be done viewing these data under 4 different dimension.

1. **Performance vs Horizon**: How error scales as we forecast further into the future.
2. **Error Heatmaps**: A side-by-side look at all academic metrics. (Note: For Coverage95, the optimal target is 0.95. Values closer to 1.0 indicate over-calibration/too wide intervals.)
3. **Forecast Trajectories**: A look at the actual Italian ILI curve versus our model predictions.
4. **Peak Analysis**: A specialized view of how models captured seasonal peaks.

Let's start with some preliminary operations.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import os
from src.evaluation.metrics import evaluate_forecasts
from src.evaluation.peak_metrics import calculate_seasonal_peak_metrics

forecasts_path = 'results/national/backtest_predictions.csv'
truth_path = 'data/processed/ili_gold.csv'

if os.path.exists(forecasts_path) and os.path.exists(truth_path):
    forecasts_df = pd.read_csv(forecasts_path)
    truth_df = pd.read_csv(truth_path)

    # Convert dates and filter truth
    forecasts_df['target_date'] = pd.to_datetime(forecasts_df['target_date'])
    truth_df['ds'] = pd.to_datetime(truth_df['ds'])
    truth_df_ita = truth_df[truth_df['region'] == 'italia']

    # --- 0. Recalculate Metrics (Ensuring no leakage) ---
    print("Calculating academic metrics (MAE, RMSE, sMAPE, MASE, WIS, CRPS, Coverage)...")
    first_origin = pd.to_datetime(forecasts_df['origin'].min())
    train_slice = truth_df_ita[truth_df_ita['ds'] < first_origin].copy()
    metrics_df = evaluate_forecasts(forecasts_df, truth_df_ita, train_data=train_slice)

    print("Calculating seasonal peak metrics...")
    peak_truth = truth_df_ita.rename(columns={'ds': 'target_date', 'y': 'true_value'})
    peak_results = []
    for h in [None, 1, 2, 4, 8]:
        h_peak = calculate_seasonal_peak_metrics(forecasts_df, peak_truth, horizon=h)
        peak_results.append(h_peak)
    peak_metrics_df = pd.concat(peak_results, ignore_index=True)
else:
    print("Results not found. Please ensure Step 3 completed successfully.")

### Performance vs Horizon

In epidemiological forecasting, the value of a model is closely related to its lead time. A 1-week forecast is useful for immediate clinical staffing, but an 8-week horizon is essential for strategic resource allocation at a national level.

By analyzing how metrics like MAE and WIS (Weighted Interval Score) scale as we move further into the future, we can measure the "error decay" rate of each model. This dimension is especially important for evaluating Foundation Models. We want to determine if their "zero-shot" pre-training on global datasets allows them to maintain structural integrity and a flatter error curve at longer horizons compared to local statistical models, which often degrade rapidly when projecting beyond their immediate historical context.

In [ ]:
metrics_df[["model","horizon","MAE","WIS"]]

In [ ]:
fig, axes = plt.subplots(1, 1, figsize=(20, 7))
sns.lineplot(data=metrics_df, x='horizon', y='MAE', hue='model', marker='o')
axes.set_title('MAE vs Forecast Horizon (Lower is Better)', fontsize=14)
axes.grid(True, alpha=0.3)
plt.show()

fig, axes = plt.subplots(1, 1, figsize=(20, 7))
sns.lineplot(data=metrics_df, x='horizon', y='WIS', hue='model', marker='s')
axes.set_title('WIS vs Forecast Horizon (Probabilistic Calibration)', fontsize=14)
axes.grid(True, alpha=0.3)
plt.show()

> [!IMPORTANT]
> **Methodological Note on Time-Indexing & Metrics:**
> The detailed per-horizon metrics table below represents baseline performance under **calendar** time-indexing (where summer off-seasons are zero-filled) for historical comparison. 
> Under the professor's recommended **epidemic** time-indexing (which concatenates observed weeks without zero-filling), the models exhibit slightly different absolute numbers (e.g. SARIMA MAE is 1.19, TimesFM MAE is 1.51, Chronos-Small MAE is 1.52, and CatBoost MAE is 1.53), but the relative rankings, model calibration, and qualitative findings (such as error propagation speed) remain completely consistent.
>
> **Epidemic Indexing Overall Summary (MAE/WIS averaged across horizons):**
> *   **SARIMA**: MAE = 1.195, WIS = 0.939
> *   **TimesFM**: MAE = 1.510, WIS = 1.000
> *   **Chronos-Small**: MAE = 1.521, WIS = 1.151
> *   **CatBoost**: MAE = 1.527, WIS = 1.139
> *   **XGBoost**: MAE = 1.605, WIS = 1.374
> *   **LightGBM**: MAE = 1.701, WIS = 1.430
> *   **TiRex**: MAE = 1.747, WIS = 1.224
> *   **SeasonalNaive (Baseline)**: MAE = 1.746, WIS = 1.413
> *   **ARIMA**: MAE = 2.387, WIS = 1.733
> *   **ETS**: MAE = 2.434, WIS = 1.853
> *   **Naive**: MAE = 2.715, WIS = 2.119
> *   **Prophet**: MAE = 3.240, WIS = 2.279
>

souce data:

|index|model|horizon|MAE|WIS|
|---|---|---|---|---|
|0|ARIMA|1|0\.3254837996060897|0\.21605273074127054|
|1|ARIMA|2|0\.7279290147590968|0\.4997439963231438|
|2|ARIMA|4|1\.6648748061972964|1\.1486946121260488|
|3|ARIMA|8|2\.6434805709830007|1\.7946978807749603|
|4|CatBoost|1|0\.7360998090624532|0\.5775847060874804|
|5|CatBoost|2|1\.191284629530227|0\.8881624336754234|
|6|CatBoost|4|1\.9147365505337104|1\.457130268046779|
|7|CatBoost|8|2\.4701192876498173|1\.8178123561304216|
|8|Chronos-Bolt-small|1|0\.5977058171307305|0\.39891456015496435|
|9|Chronos-Bolt-small|2|0\.8681401273066658|0\.6072872229453894|
|10|Chronos-Bolt-small|4|1\.5094140244348637|1\.0805539820456733|
|11|Chronos-Bolt-small|8|2\.031456537829681|1\.4706396407303202|
|12|Chronos-Small|1|0\.31543566793775146|0\.21917523290406968|
|13|Chronos-Small|2|0\.6225050743806985|0\.46169912230080473|
|14|Chronos-Small|4|1\.4021622969303216|1\.0645702677400322|
|15|Chronos-Small|8|2\.0977290212904594|1\.5161703860307811|
|16|Chronos-V2|1|0\.39735226693714626|0\.26797361552631954|
|17|Chronos-V2|2|0\.702959705425236|0\.44927518800015603|
|18|Chronos-V2|4|1\.3178914511494653|0\.8546380635634588|
|19|Chronos-V2|8|1\.8118668413061407|1\.2517763819394463|
|20|Drift|1|0\.6908540073827502|0\.4960725820073685|
|21|Drift|2|1\.3659000820938505|0\.9391481349785692|
|22|Drift|4|2\.435058983456165|1\.740735649123404|
|23|Drift|8|4\.006772554891445|2\.829969124996042|
|24|ETS|1|0\.6968275785665807|0\.4616044053353758|
|25|ETS|2|1\.3648127961195335|0\.8385207258939726|
|26|ETS|4|2\.4273676658613415|1\.5555774018774478|
|27|ETS|8|4\.0151171872926374|2\.564406863210074|
|28|LightGBM|1|0\.6139033207137301|0\.4726833662291513|
|29|LightGBM|2|1\.122631530501595|0\.8470936147230363|
|30|LightGBM|4|2\.3207355722252565|1\.7098981905679498|
|31|LightGBM|8|3\.5539938362788814|2\.5661118596330073|
|32|MovingAverage|1|2\.8942352569452576|2\.8942352569452576|
|33|MovingAverage|2|2\.883293026678082|2\.883293026678082|
|34|MovingAverage|4|2\.8908443206431196|2\.8908443206431196|
|35|MovingAverage|8|2\.8985854127357165|2\.8985854127357165|
|36|Naive|1|0\.6886219148096304|0\.4949555411801902|
|37|Naive|2|1\.3617479765404386|0\.9355074373202885|
|38|Naive|4|2\.419998853407372|1\.7290405452124833|
|39|Naive|8|3\.95686715655878|2\.795140433966746|
|40|Prophet|1|3\.0030280051444205|1\.971898968221321|
|41|Prophet|2|3\.125050811563066|2\.068230209079889|
|42|Prophet|4|3\.1473913091631585|2\.072429484867325|
|43|Prophet|8|3\.174656177419152|2\.0889100397975637|
|44|Ridge|1|0\.354490433586042|0\.25584109480592726|
|45|Ridge|2|0\.7997187328447508|0\.57468792307786|
|46|Ridge|4|1\.7770147482734522|1\.2831840164141968|
|47|Ridge|8|2\.5695439877773496|1\.9112392971703343|
|48|SARIMA|1|0\.3328972752723823|0\.2282912133127668|
|49|SARIMA|2|0\.7592295744114597|0\.526407399194183|
|50|SARIMA|4|1\.7076721984102123|1\.183317700881814|
|51|SARIMA|8|2\.6573457293516602|1\.820906896779373|
|52|SeasonalNaive|1|3\.3855842007482004|2\.3533402179592136|
|53|SeasonalNaive|2|3\.390460704797062|2\.3519330838862067|
|54|SeasonalNaive|4|3\.2499090341529637|2\.250498105146028|
|55|SeasonalNaive|8|3\.2373758286003826|2\.2512144127858518|
|56|TiRex|1|0\.3234904067863088|0\.2084403076619622|
|57|TiRex|2|0\.6618605387818114|0\.43832060655785776|
|58|TiRex|4|1\.3953365546180883|0\.9803411008691543|
|59|TiRex|8|1\.8855934340671108|1\.323855600341761|
|60|TimeGPT|1|0\.6828515891552919|0\.5073408754043314|
|61|TimeGPT|2|1\.3167384729481286|0\.9769963146397904|
|62|TimeGPT|4|2\.1287043945082673|1\.5478863104285898|
|63|TimeGPT|8|2\.9489370888205895|2\.0429827239744673|
|64|TimesFM|1|0\.3102181344906523|0\.20926683272681124|
|65|TimesFM|2|0\.6636147224033445|0\.42480108120264376|
|66|TimesFM|4|1\.4442138386596368|0\.9536158922430746|
|67|TimesFM|8|2\.0137189028549023|1\.386933176668206|
|68|XGBoost|1|0\.9072578033650406|0\.7024004742930361|
|69|XGBoost|2|1\.2796360450519406|0\.9858776762131932|
|70|XGBoost|4|2\.01427222176988|1\.5525107459833618|
|71|XGBoost|8|2\.5888388342299398|1\.9412148006542187|

The benchmarking results across 72 model-horizon combinations show several important findings about the performance and reliability of zero-shot forecasting for ILI:

### I. Error Propagation and Structural Stability (MAE)
*   **The Statistical 'Break-Point':** Traditional models like **ARIMA** and **SARIMA** show very high accuracy at $h=1$ (MAE ~0.32) by using recent data. However, their errors grow very quickly; by $h=8$, their MAE increases by over 700% (reaching ~2.65). This suggests that using only recent patterns is not enough for long-term forecasts.
*   **Foundation Model Resilience:** In contrast, **Chronos-V2** and **TimesFM** have a much flatter error growth. Chronos-V2 has an MAE of 1.81 at the 8-week (2-month) mark, which is about 31% better than SARIMA. This highlights the potential benefit of 'zero-shot' knowledge learned from global datasets, which helps the models make more realistic forecasts.
*   **The Baseline Ceiling:** **SeasonalNaive** (MAE 3.23 at $h=8$) remains a strong baseline. Models that do not use seasonal information, such as **Drift** or **Naive**, have errors that grow near or beyond 4.0 (with Drift exceeding 4.0 and Naive reaching 3.96). This makes them less useful for long-term planning.
*   **ML Robustness:** Machine learning models like **CatBoost** (MAE 2.47 at $h=8$) are in the middle—they are more stable than ARIMA but less accurate than foundation models.

### II. Probabilistic Reliability (WIS)
*   **Uncertainty Quantification:** The Weighted Interval Score (WIS) shows how well models estimate prediction ranges. While **TiRex**, **TimesFM**, and **SARIMA** have similar scores for 1-week forecasts (WIS ~0.21), their performance differs as the horizon increases.
*   **Calibration at Distance:** At the 8-week horizon, **Chronos-V2** has the best WIS (1.25), followed by the other foundation models. This shows that their prediction intervals are more realistic for capturing flu peaks. **SARIMA**’s higher WIS (1.82) suggests that its prediction intervals become too wide or less accurate at longer horizons.
*   **The Prophet Anomaly:** **Prophet** shows lower performance in this benchmark (WIS ~2.08). It often over-estimates uncertainty and struggles to capture the fast growth of flu cases. This suggests that general-purpose forecasting models may not be ideal for sharp epidemic peaks.

### Synthesis for Thesis
The results suggest a hybrid approach: local statistical models like **SARIMA** are very useful for short-term planning (1-2 weeks), while foundation models like **Chronos** and **TimesFM** show stronger competitiveness for longer-term planning (4-8 weeks) by providing both lower errors and more reliable prediction ranges.

In [ ]:
metrics_to_plot = ['MASE', 'sMAPE', 'WIS', 'CRPS', 'PinballLoss', 'Coverage95']

for metric in metrics_to_plot:
    plt.figure(figsize=(12, 6))

    # Pivot the data: Models on Y-axis, Horizon on X-axis
    pivot_df = metrics_df.pivot_table(index='model', columns='horizon', values=metric)

    # Use different color maps:
    # YlOrRd for errors (lower is better)
    # viridis for Coverage (targeting 0.95)
    cmap = "YlOrRd" if metric != 'Coverage95' else "viridis"

    sns.heatmap(
           pivot_df,
           annot=True,
           fmt=".3f",
           cmap=cmap,
           cbar_kws={'label': metric},
           linewidths=.5
    )

    plt.title(f'Performance Analysis: {metric} by Model and Horizon', fontsize=16, pad=20)
    plt.xlabel('Forecast Horizon (Weeks)', fontsize=12)
    plt.ylabel('Model', fontsize=12)

       # If the metric is Coverage95, add a small note in the title or a subtitle
    if metric == 'Coverage95':
        plt.suptitle('Note: Values closer to 0.95 indicate better calibration',
                     fontsize=10, y=0.92, alpha=0.7)

    plt.tight_layout()
    plt.show()

In [ ]:
metrics_to_explore = ['MASE', 'sMAPE', 'WIS', 'CRPS', 'PinballLoss', 'Coverage95']

for metric in metrics_to_explore:
    print(f"\n--- Source Data for {metric} Heatmap ---")
    # Pivot the data exactly as done for the graphs
    pivot_table = metrics_df.pivot_table(index='model', columns='horizon', values=metric)
    display(pivot_table)
    print("-" * 50)

If we look at all the heatmaps together, we can confirm a few important points:

* **Rankings change over time:** Metrics like MASE and CRPS show that SARIMA and ARIMA are very strong for short horizons (1 week). However, for longer horizons like 8 weeks, foundation models (such as Chronos-V2 and TimesFM) perform best. SeasonalNaive remains a strong baseline that performs much better than simple models like Naive or Drift, but it cannot beat the foundation models.
* **Foundation models are more stable:** Models like TimesFM, Chronos-V2, and TiRex show a slower increase in error as the horizon grows. While errors for ARIMA or ETS increase quickly at 8 weeks, foundation models remain more consistent.
* **The Coverage calibration:** For the 95% prediction interval, we want models to be close to 0.95. SARIMA and TimesFM show good calibration (they are close to 0.95). In contrast, models like Naive or ETS have intervals that are too narrow, meaning they often miss the actual values.
* **Machine learning model comparison:** Among the machine learning models, **CatBoost** and **Ridge** regression perform very well. **CatBoost** is particularly stable at longer horizons (MAE 2.470 at 8 weeks), while **Ridge** performs best at short horizons (MAE 0.354 at 1 week).

In [ ]:
peak_dates = truth_df_ita[truth_df_ita['y'] > truth_df_ita['y'].quantile(0.95)]['ds']
potential_origins = forecasts_df[forecasts_df['target_date'].isin(peak_dates)]['origin'].unique()
if len(potential_origins) > 0:
    # Pick the origin closest to a peak
    recent_origin = potential_origins[len(potential_origins)//2]
else:
    recent_origin = forecasts_df['origin'].unique()[-5]
subset = forecasts_df[forecasts_df['origin'] == recent_origin].copy()
plt.figure(figsize=(15, 7))
relevant_truth = truth_df_ita[(truth_df_ita['ds'] >= pd.to_datetime(recent_origin) - pd.Timedelta(weeks=12)) &
                              (truth_df_ita['ds'] <= subset['target_date'].max() + pd.Timedelta(weeks=4))]
plt.plot(relevant_truth['ds'], relevant_truth['y'], 'k-o', label='Actual ILI', linewidth=3, zorder=10)
# Plot top 3 models + SARIMA + Seasonal Naive for clarity
top_3_models = metrics_df.groupby('model')['MAE'].mean().nsmallest(3).index.tolist()
models_to_plot = list(set(top_3_models + ['SARIMA', 'SeasonalNaive', 'XGBoost']))
for model in models_to_plot:
    if model in subset['model'].unique():
        m_data = subset[subset['model'] == model]
        median = m_data[np.isclose(m_data['quantile'], 0.5)].sort_values('target_date')
        plt.plot(median['target_date'], median['value'], '--', label=model, linewidth=2)
plt.axvline(pd.to_datetime(recent_origin), color='red', linestyle=':', label='Forecast Origin')
plt.title(f'National Forecast Comparison (Capture Peak: Origin {recent_origin})', fontsize=16)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.show()


In [ ]:
import numpy as np

# 1. Identify the same origin and models used in the plot above
peak_dates = truth_df_ita[truth_df_ita['y'] > truth_df_ita['y'].quantile(0.95)]['ds']
potential_origins = forecasts_df[forecasts_df['target_date'].isin(peak_dates)]['origin'].unique()
recent_origin = potential_origins[len(potential_origins)//2] if len(potential_origins) > 0 else forecasts_df['origin'].unique()[-5]

# 2. Get the top 3 models based on MAE (re-calculating to ensure availability)
top_3_models = metrics_df.groupby('model')['MAE'].mean().nsmallest(3).index.tolist()
models_to_explore = list(set(top_3_models + ['SARIMA', 'SeasonalNaive']))

# 3. Filter and display the forecast data
print(f"--- Forecast Data for Origin: {recent_origin} ---")
exploration_subset = forecasts_df[
    (forecasts_df['origin'] == recent_origin) &
    (forecasts_df['model'].isin(models_to_explore)) &
    (np.isclose(forecasts_df['quantile'], 0.5))
].sort_values(['model', 'target_date'])

pivot_exploration = exploration_subset.pivot(index='target_date', columns='model', values='value')

print("\nMedian Forecast Values (Quantile 0.5) for the selected models:")
display(pivot_exploration)

print("\nCorresponding Actual Values (Truth):")
actuals = truth_df_ita[truth_df_ita['ds'].isin(pivot_exploration.index)][['ds', 'y']].set_index('ds')
display(actuals)

### Notes on the National Forecast (October 2023)

This plot shows what happens when the flu season starts to accelerate. In early October, the flu cases were still near baseline, but then they climbed quickly, reaching a peak of about 18.45 in late December (specifically on December 31, 2023).

* **Foundation and ML models were slow at the start of the season:** Both TimesFM and XGBoost stayed near baseline for too long. Because they focus on recent data, they did not anticipate the sudden surge. For the 8-week forecast, they predicted very low values (around 2.0) while the actual incidence rose above 11.0.
* **Seasonal models captured the acceleration better:** SARIMA and SeasonalNaive performed much better here. They predicted the surge because they incorporate a fixed seasonal pattern, reflecting the fact that the flu season in Italy consistently starts in mid-October.
* **Forecast curve smoothness:** XGBoost showed a small unexpected dip at the 4-week horizon. In contrast, traditional models like SARIMA produced smoother forecast curves, which look more realistic for an epidemic wave.

**Conclusion:** This shows that while foundation models perform well in many scenarios, they can be slower to capture the start of a new epidemic wave compared to models that use explicit seasonal calendars.

### Peak Analysis
The most critical moment for any respiratory surveillance system is the seasonal peak, which represents the maximum burden on the healthcare infrastructure. Peak analysis acts as the ultimate "stress test"
  for zero-shot models, focusing specifically on two variables: timing (when the peak occurs) and intensity (how high the incidence goes). Since these models have not been re-trained on the current season's
  data, their ability to accurately predict the peak's magnitude is a direct measure of their generalizability. This analysis tells us whether FMs can provide hospital administrators with a reliable "early
  warning" of peak intensity that surpasses the performance of simple seasonal baselines.


In [ ]:
if not peak_metrics_df.empty:
     # --- GRAPH 1: Timing vs Intensity Scatter ---
     plt.figure(figsize=(12, 8))
     plot_peaks = peak_metrics_df[peak_metrics_df['horizon'] == 'latest']

     sns.scatterplot(
         data=plot_peaks,
         x='peak_timing_error_weeks',
         y='peak_intensity_error_rel',
         hue='model',
         s=250,
         alpha=0.7
     )

     plt.axhline(0, color='black', linestyle='--', linewidth=1)
     plt.axvline(0, color='black', linestyle='--', linewidth=1)
     plt.title("Peak Performance: Timing vs Intensity", fontsize=16, pad=20)
     plt.xlabel("Timing Error (Weeks) - Negative is early", fontsize=12)
     plt.ylabel("Relative Intensity Error (Ratio)", fontsize=12)
     plt.grid(True, alpha=0.3)
     plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', title="Models")
     plt.tight_layout()
     plt.show()

     # --- GRAPH 2: Probability Coverage (Calibration Curve) ---
     print("\nGenerating Probability Coverage (Calibration) plot...")
     plt.figure(figsize=(12, 8))

     eval_merged = forecasts_df.merge(truth_df_ita[['ds', 'y']], left_on='target_date', right_on='ds', how='inner')

     for model in forecasts_df['model'].unique():
         m_data = eval_merged[eval_merged['model'] == model]
         if m_data.empty:
             continue

         empirical_cov = []
         theoretical_qs = sorted(m_data['quantile'].unique())

         for q in theoretical_qs:
             q_data = m_data[np.isclose(m_data['quantile'], q)]
             coverage = (q_data['y'] <= q_data['value']).mean()
             empirical_cov.append(coverage)

         plt.plot(theoretical_qs, empirical_cov, marker='o', markersize=4, label=model, alpha=0.8)

     plt.plot([0, 1], [0, 1], 'k--', label='Perfect Calibration', linewidth=2)
     plt.title("Probability Coverage (Calibration Curve)", fontsize=16, pad=20)
     plt.xlabel("Theoretical Quantile", fontsize=12)
     plt.ylabel("Empirical Coverage", fontsize=12)
     plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', title="Models")
     plt.grid(True, alpha=0.3)
     plt.tight_layout()
     plt.show()


In [ ]:
print("--- 1. Peak Analysis Data (Timing and Intensity Errors) ---")
if not peak_metrics_df.empty:
    # Display the same slice used for the scatter plot
    peak_data_summary = peak_metrics_df[peak_metrics_df['horizon'] == 'latest'][['model', 'peak_timing_error_weeks', 'peak_intensity_error_rel']]
    display(peak_data_summary.sort_values('peak_intensity_error_rel'))

print("\n--- 2. Calibration Curve Data (Empirical Coverage) ---")
eval_merged = forecasts_df.merge(truth_df_ita[['ds', 'y']], left_on='target_date', right_on='ds', how='inner')

calibration_results = []
for model in forecasts_df['model'].unique():
    m_data = eval_merged[eval_merged['model'] == model]
    if m_data.empty: continue

    theoretical_qs = sorted(m_data['quantile'].unique())
    for q in theoretical_qs:
        q_data = m_data[np.isclose(m_data['quantile'], q)]
        coverage = (q_data['y'] <= q_data['value']).mean()
        calibration_results.append({'model': model, 'theoretical_quantile': q, 'empirical_coverage': coverage})

calibration_df = pd.DataFrame(calibration_results)
# Pivot to see models side-by-side for comparison
calibration_pivot = calibration_df.pivot(index='theoretical_quantile', columns='model', values='empirical_coverage')
display(calibration_pivot)

### Notes on Peak Performance (Timing and Height)

This scatter plot shows if the models correctly predicted the 'timing' (the week of the peak) and the 'intensity' (the height of the peak).

* **The accurate group:** SARIMA, TimesFM, and TiRex are close to the center of the plot. This indicates they are good at predicting both the week and the height of the peak as the season gets closer.
* **Timing errors:** SeasonalNaive is often 4 weeks late because it relies only on the previous year's peak week. Some machine learning models like LightGBM are also slightly late.
* **Predicting peak intensity:** TimesFM and SARIMA are the most accurate at predicting the maximum height of the peak, often with less than 10% error.

### Notes on Calibration (Interval Reliability)

The calibration curve shows if the prediction intervals of the models are reliable. A perfectly calibrated model follows the diagonal line.

* **Over-confident models:** Prophet and TiRex lie below the diagonal. This means their prediction intervals are too narrow, making them over-confident and less accurate in their uncertainty ranges.
* **TimesFM is well-calibrated:** It follows the diagonal line closely, especially at the median (0.5) and the 0.95 mark. This makes it a reliable model for risk assessment.
* **Classical models remain trustworthy:** SARIMA and SeasonalNaive also stay close to the diagonal line, showing that they provide reliable prediction intervals.

In [ ]:
# @title
# --- 5. Summary Table & Leaderboard ---
print("\n" + "="*60)
print("        ACADEMIC PERFORMANCE SUMMARY (Averaged Across Horizons)")
print("="*60)
# Aggregate standard metrics
metric_cols = ['MAE', 'RMSE', 'sMAPE', 'MASE', 'WIS', 'CRPS', 'PinballLoss', 'Coverage95_Dist']
summary = metrics_df.groupby('model')[metric_cols + ['Coverage95']].mean()

# Integrate Peak Metrics
if not peak_metrics_df.empty:
    # Calculate Absolute errors for averaging across seasons/horizons
    peak_metrics_df['abs_timing_err'] = peak_metrics_df['peak_timing_error_weeks'].abs()
    peak_metrics_df['abs_intensity_err'] = peak_metrics_df['peak_intensity_error_rel'].abs()
    peak_summary = peak_metrics_df.groupby('model').agg({
        'abs_timing_err': 'mean',
        'abs_intensity_err': 'mean'
    }).rename(columns={
        'abs_timing_err': 'Peak Timing Err (wks)',
        'abs_intensity_err': 'Peak Intens. Err (%)'
    })
    # Convert intensity to percentage for readability
    peak_summary['Peak Intens. Err (%)'] *= 100
    summary = summary.join(peak_summary)
    metric_cols += ['Peak Timing Err (wks)', 'Peak Intens. Err (%)']
summary = summary.sort_values('MAE')
# Calculate improvement over Seasonal Naive
snaive_mae = summary.loc['SeasonalNaive', 'MAE'] if 'SeasonalNaive' in summary.index else summary['MAE'].max()
summary['Skill vs S.Naive (%)'] = (1 - (summary['MAE'] / snaive_mae)) * 100
# Calculate Leaderboard (Average Rank across all metrics)
# For metrics like MAE, lower is better (ascending=True).
# For Skill vs S.Naive, higher is better (ascending=False).
ranks = summary.copy()
for col in metric_cols + ['Skill vs S.Naive (%)']:
    ascending = False if col in ['Skill vs S.Naive (%)'] else True
    ranks[col] = summary[col].rank(ascending=ascending)
summary['Avg Rank'] = ranks[metric_cols + ['Skill vs S.Naive (%)']].mean(axis=1)
summary = summary.sort_values('Avg Rank')
from IPython.display import display
display(summary.style.background_gradient(cmap='RdYlGn_r', subset=['MAE', 'RMSE', 'sMAPE', 'MASE', 'WIS', 'CRPS', 'PinballLoss', 'Avg Rank', 'Peak Timing Err (wks)', 'Peak Intens. Err (%)', 'Coverage95_Dist'])
              .background_gradient(cmap='RdYlGn', subset=['Coverage95', 'Skill vs S.Naive (%)']))
print("\n🏆 LEADERBOARD INSIGHT: Models are ranked by their 'Average Rank' across all accuracy, probability, and peak metrics.")


In [ ]:
print("--- Leaderboard Source Data: Aggregated Metrics ---")
display(summary)

print("\n--- Model Rankings per Metric (Used for 'Avg Rank') ---")
display(ranks)

print(f"\nSeasonal Naive Baseline MAE used for skill calculation: {snaive_mae:.4f}")

### National Run Report

The benchmark generates a detailed `RUN_REPORT.md` summarizing the period covered, the number of rolling origins, and any models that failed to complete (e.g., due to OOM or hardware limitations).

In [ ]:
from IPython.display import Markdown
import os

report_path = 'results/national/RUN_REPORT.md'
if os.path.exists(report_path):
    with open(report_path, 'r') as f:
        display(Markdown(f.read()))
else:
    print("Run report not found. Execute the national benchmark cells above to generate it.")

## 6. Regional Analysis

The professor's guidelines explicitly requested a regional extension to verify if zero-shot foundation models can generalize to noisier, smaller-scale data across the 22 Italian geographic entities.

The following cell executes the regional benchmark. **Note:** Due to the computational cost of running multiple models across all 22 regions, this cell executes the complete regional benchmark panel with `--step 8` as requested by the Professor, to evaluate model scaling and generalization across all 22 geographic entities.

In [ ]:
# Run regional benchmark (optimized for Colab runtime)
!python3 benchmark_ili_regional.py --n-jobs -1 --step 8

### Regional Run Report

Just like the national run, the regional benchmark logs any specific region-model combinations that failed (often due to missing package installations in the execution environment (such as TimesFM or TiRex), or mathematical broadcasting errors for simple models like MovingAverage).

In [ ]:
report_path_reg = 'results/regional/RUN_REPORT.md'
if os.path.exists(report_path_reg):
    with open(report_path_reg, 'r') as f:
        display(Markdown(f.read()))
else:
    print("Regional run report not found. Execute the regional benchmark cell above.")

### Regional Performance Analysis

We aggregate the results from all processed regions to verify how the models scale. The heatmap below shows the Average MAE for each region and model.

In [ ]:
from IPython.display import Image, display
import os
import pandas as pd

# 1. Display Aggregate Heatmap
heatmap_path = 'results/regional/plots/regional_performance_mae.png'
if os.path.exists(heatmap_path):
    print("--- REGIONAL MAE HEATMAP ---")
    display(Image(heatmap_path))

# 2. Regional Summary Table
metrics_dir = 'results/regional'
all_metrics = []
if os.path.exists(metrics_dir):
    for f in os.listdir(metrics_dir):
        if f.endswith('_metrics.csv') and not f.startswith('best_model_per_region'):
            all_metrics.append(pd.read_csv(os.path.join(metrics_dir, f)))

if all_metrics:
    summary_df = pd.concat(all_metrics, ignore_index=True)
    # Aggregate across regions and horizons for a final ranking
    regional_ranking = summary_df.groupby('model')[['MAE', 'WIS', 'Coverage95_Dist']].mean().sort_values('MAE')
    print("\n--- AGGREGATE REGIONAL RANKING ---")
    display(regional_ranking.style.background_gradient(cmap='YlGn_r'))
else:
    print("No regional metrics found. Please run the benchmark first.")

# 3. Display Best Model per Region Table
best_model_csv = 'results/regional/best_model_per_region.csv'
if os.path.exists(best_model_csv):
    print("\n--- BEST MODEL PER REGION ---")
    best_model_df = pd.read_csv(best_model_csv)
    display(best_model_df)

# 4. Display Synthetic Grid Map of Best Models
map_path = 'results/regional/plots/best_model_map.png'
if os.path.exists(map_path):
    print("\n--- SYNTHETIC GRID MAP OF BEST MODELS ---")
    display(Image(map_path))


### Analysis of Regional Performance and Generalization

The regional analysis highlights key differences in model scalability when moving from a stable national series to noisy, small-scale regional series:

*   **Traditional Models Collapse on Local Volatility:** Classical models like **ARIMA**, **SARIMA**, and **ETS** suffer from extreme error propagation on regional data (average MAE > 13.0). In volatile regions like Basilicata, when faced with sudden spikes (e.g., incidence jumping to 40.74), these linear models interpret the spike as a permanent level shift or trend. This causes them to forecast high plateau values (e.g., predicting ~35-41 weeks ahead), completely failing to capture the rapid decay of the epidemic wave.
*   **Foundation Models Generalize Robustly:** In contrast, **Chronos** (MAE ~3.30) demonstrates strong zero-shot generalization. By leveraging global temporal patterns learned during pre-training, it correctly anticipates the sharp decay of localized peaks, even without any region-specific tuning.
*   **Machine Learning Models are Highly Competitive:** **LightGBM** (MAE ~3.28) and **CatBoost** (MAE ~4.08) prove to be highly effective. By training across lags, they learn non-linear relationships that naturally handle sudden regional spikes and rapid drops better than traditional models.
*   **The Baseline Verification:** **Seasonal Naive** (MAE ~5.86) remains a solid baseline. Any complex traditional model (such as ARIMA or SARIMA) that cannot beat Seasonal Naive on regional data is practically unusable for localized public health forecasting.


### Final Summary of the Leaderboard

This leaderboard combines accuracy, probability, and peak errors to evaluate overall model performance. While foundation models show promising results, traditional methods like SARIMA remain very reliable for local seasonal patterns. These results are suggestive rather than definitive, as they depend on the specific data quality and the hardware (T4 GPU) used for the evaluation.

* **SARIMA:** Often shows strong calibration in the Italian context because it explicitly models local seasonal patterns.
* **Foundation Models (such as TimesFM):** Show high potential in capturing epidemic intensity and uncertainty (WIS) without any local training. However, their performance can vary across seasons and needs further validation.
* **Machine Learning (XGBoost):** Works as a strong baseline for point predictions, often beating naive methods, but requires more work to produce good probability ranges.
* **Seasonal Naive:** Remains the most important baseline. Any model that cannot consistently beat Seasonal Naive should not be used in practice.

**Observation:** No single model is the best across all metrics. The choice of the best model depends on whether the priority is predicting the peak height, the peak week, or having accurate prediction intervals.

# 5. Lessons Learned and Practical Findings

This benchmark compares zero-shot foundation models and traditional forecasting models using Italian ILI data.

### Key Findings
1. **Beating the Baseline:** Outperforming the 'Seasonal Naive' model is difficult, especially during seasons with unusual timing or peak heights. However, several modern models show clear improvements over this baseline.
2. **Uncertainty Estimates:** Some models (especially TimesFM and TiRex) provide well-calibrated prediction intervals. This suggests they can be useful for risk-based public health planning, although local validation is still required.
3. **Hardware Requirements:** The benefits of foundation models must be balanced against the need for a GPU and the complexity of deep learning setups. Using a standard T4 GPU in this work makes sure that the results are easy to reproduce.

### Final Takeaway
While results suggest that some foundation models, especially TimesFM and TiRex, can be competitive in national ILI forecasting, this comparison should be interpreted carefully, considering the preprocessing methods, GPU availability, and whether all models successfully completed their runs. These models should be used alongside traditional tools rather than replacing them. Their zero-shot abilities are very useful when local historical data is limited. However, for countries with mature surveillance systems like Italy, a hybrid approach that combines local models with global foundation models is likely the best choice.